# TP 2 — Construire l'auto-attention pas à pas

**Objectif :** partir d'une simple moyenne des tokens précédents et arriver à une tête qui apprend elle-même quels tokens doivent contribuer.

Boucles (moyenne causale) → produit matriciel (poids fixes) → masque + Softmax (poids normalisés) → self-attention (poids appris).

**Invariant à suivre :** une ligne de poids contient la contribution des tokens passés au calcul du token courant, et sa somme vaut toujours 1.

**Livrable :** les quatre versions, les tests `torch.allclose`, la matrice `wei[0]` et une explication des formes B, T, C, d_h.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Agréger par produit matriciel

Une ligne de matrice peut choisir et pondérer plusieurs vecteurs : $c_t = \sum_i a_{t,i}\, b_i$.

In [2]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)

b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)
print("Somme des lignes de a :", a.sum(1))

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
Somme des lignes de a : tensor([1., 1., 1.])


### Questions d'interprétation

**Sans Python, calculez c₂.**
*Réponse :* la 2ᵉ ligne de `a` vaut (1/2, 1/2, 0), donc c₂ = (b₁ + b₂)/2 : la moyenne, composante par composante, des deux premières lignes de `b`

**Pourquoi la somme de chaque ligne de `a` vaut-elle 1 ?**
*Réponse :* on a divisé chaque ligne de la matrice triangulaire par son nombre de 1 (`keepdim=True` permet le broadcasting ligne par ligne). Chaque ligne devient une moyenne : des poids positifs de somme 1.

## Étape 2 — Moyenner uniquement le passé

Pour chaque batch `b` et chaque instant `t`, on construit la moyenne de tous les tokens visibles jusqu'à `t`. Aucune position ne doit accéder au futur.

In [3]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]               # (t+1, C)
        xbow[b, t] = torch.mean(xprev, dim=0)

for t in (0, 3, 7):
    print(t, x[0, :t + 1].shape)
print(x[0])
print(xbow[0])

0 torch.Size([1, 2])
3 torch.Size([4, 2])
7 torch.Size([8, 2])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


### Questions d'interprétation

**Formes de `xprev` :** (1, 2) pour t = 0, (4, 2) pour t = 3, (8, 2) pour t = 7.

**Pourquoi la moyenne porte-t-elle sur `dim=0` ?**
*Réponse :* dans `xprev` (forme `(t+1, C)`), la dimension 0 est le temps. On moyenne à travers les tokens passés, en gardant séparées les C composantes de chaque vecteur.

## Étapes 3–4 — Vectoriser

La matrice triangulaire reproduit la moyenne causale ; le masque −∞ suivi de Softmax produit exactement les mêmes poids.

In [4]:
# Version 2 : matrice triangulaire normalisée
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x            # (T,T) @ (B,T,C) -> (B,T,C)
print(torch.allclose(xbow, xbow2, atol=1e-6))

True


In [5]:
# Version 3 : masque -inf puis Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow, xbow3, atol=1e-6))
print(wei)

True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


### Questions d'interprétation

**Pourquoi `wei @ x` fonctionne-t-il avec une matrice (T, T) et un tenseur (B, T, C) ?**
*Réponse :* le produit matriciel agit sur les deux dernières dimensions et diffuse (broadcast) les autres : `wei` est traitée comme `(1, T, T)` puis répétée pour chacun des B éléments du batch ; on calcule (T,T) @ (T,C) = (T,C) pour chaque b.

**Que devient Softmax aux emplacements contenant −∞ ?**
*Réponse :* exp(−∞) = 0, donc le poids vaut exactement 0 : le futur ne contribue pas. Les positions restantes (toutes à 0) reçoivent des poids égaux, d'où la moyenne uniforme.

## Étape 5 — Calculer des poids d'attention

Les poids ne sont plus fixés à 1/(t+1) : ils proviennent de la compatibilité entre les *queries* et les *keys*.

In [6]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)                                   # (B, T, 16)
q = query(x)                                 # (B, T, 16)
wei = q @ k.transpose(-2, -1)                # (B, T, 16) @ (B, 16, T) -> (B, T, T)
wei = wei * head_size**-0.5                  # mise à l'échelle
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)                                 # (B, T, 16)
out = wei @ v                                # (B, T, T) @ (B, T, 16) -> (B, T, 16)

print("k, q, v :", k.shape, q.shape, v.shape)
print("wei     :", wei.shape)
print("out     :", out.shape)

k, q, v : torch.Size([4, 8, 16]) torch.Size([4, 8, 16]) torch.Size([4, 8, 16])
wei     : torch.Size([4, 8, 8])
out     : torch.Size([4, 8, 16])


In [7]:
torch.set_printoptions(precision=3, sci_mode=False)
print(wei[0])

tensor([[1.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.397, 0.603, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.307, 0.289, 0.404, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.323, 0.218, 0.244, 0.215, 0.000, 0.000, 0.000, 0.000],
        [0.148, 0.203, 0.166, 0.146, 0.337, 0.000, 0.000, 0.000],
        [0.126, 0.249, 0.132, 0.106, 0.314, 0.072, 0.000, 0.000],
        [0.160, 0.199, 0.114, 0.112, 0.142, 0.167, 0.106, 0.000],
        [0.084, 0.120, 0.108, 0.154, 0.109, 0.115, 0.156, 0.155]],
       grad_fn=<SelectBackward0>)


In [8]:
# Vérifications de l'invariant
print("Somme des lignes = 1 :", torch.allclose(wei.sum(-1), torch.ones(B, T)))
futur = torch.triu(torch.ones(T, T), diagonal=1).bool()
print("Aucun poids vers le futur :", (wei[:, futur] == 0).all().item())

Somme des lignes = 1 : True
Aucun poids vers le futur : True


### Questions d'interprétation

**Quelle ligne de `wei[0]` correspond au dernier token ?**
*Réponse :* la dernière ligne, `wei[0, -1]` (index T−1 = 7). C'est la seule qui voit les 8 positions ; ses poids ne sont plus uniformes mais dépendent des données.

**Pourquoi la sortie mélange-t-elle `v`, et non directement `k` ?**
*Réponse :* les rôles sont séparés. `q` exprime « ce que je cherche », `k` « ce que je contiens, pour être trouvé », et `v` « ce que je transmets si l'on me sélectionne ». Les keys servent uniquement à calculer la compatibilité ; une projection distincte pour les valeurs laisse le modèle transmettre une information différente de celle qui sert à être repéré.

**Formes :** B = batch, T = positions (temps), C = dimension des vecteurs d'entrée, d_h = `head_size`, dimension des q, k, v et de la sortie de la tête.

## Étape 6 — Stabiliser Softmax

La variance du produit scalaire augmente avec d_h. Diviser par √d_h ramène les scores autour d'une variance 1.

In [9]:
torch.manual_seed(1337)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

raw = q @ k.transpose(-2, -1)
scaled = raw * head_size**-0.5

print("Var(k), Var(q)  :", k.var().item(), q.var().item())
print("Var brute       :", raw.var().item())
print("Var mise à l'échelle :", scaled.var().item())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

Var(k), Var(q)  : 1.0877034664154053 1.05942964553833
Var brute       : 18.44363021850586
Var mise à l'échelle : 1.1527268886566162
tensor([0.192, 0.143, 0.235, 0.143, 0.287])
tensor([0.033, 0.003, 0.162, 0.003, 0.800])


Sans normalisation, une valeur domine rapidement et les autres poids deviennent presque nuls : Softmax tend vers un *one-hot*, chaque token n'agrège plus qu'un seul autre token et les gradients deviennent très faibles.

## Bilan du TP 2

- **Somme des lignes de `wei` :** 1.
- **Absence de poids vers le futur :** tous les coefficients au-dessus de la diagonale sont nuls (vérifié).
- **Forme finale :** `out` est de forme `(B, T, d_h)` = `(4, 8, 16)`.
- **Rôles de Q, K, V :** la query d'un token est comparée aux keys des tokens visibles ; le produit scalaire mis à l'échelle, masqué et passé par Softmax donne des poids d'attention, qui servent à faire une moyenne pondérée des values.